In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────────
import pickle
from pathlib import Path
from time import perf_counter as pc

# ─── Scientific Computing ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Geospatial Processing ──────────────────────────────────────────────────────
import geopandas as gpd
import pyproj
from scipy.spatial import cKDTree
from shapely.geometry import LineString, MultiLineString, Point

# ─── Mapping and Visualization ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import folium

# ─── Custom Modules ────────────────────────────────────────────────────────────
import geo_util as gu

In [ ]:
# global definitions
_data_path = Path( './data/' )

In [ ]:
nodes_and_edges_path = _data_path.joinpath('nodes_and_edges_all_modalities').with_suffix('.pkl')

with open(nodes_and_edges_path, 'rb') as f:
    (nl_nodes, nl_edges) = pickle.load(f)

In [ ]:
nl_nodes = gu.assignComponentsToNodes(nl_nodes, nl_edges)

In [ ]:
gu.showNodesColoredPerComponentWithBasemap(nl_nodes, width=800, height=600, file_name='before_connecting_components_with_ferries.png')

In [ ]:
ferries_path = _data_path.joinpath('ferries_nl.pkl')

with open(ferries_path, 'rb') as f:
    nl_ferries = pickle.load(f)

In [ ]:
nl_ferries_projected = nl_ferries.to_crs(gu.projectedCrsNL)
nl_nodes_projected = nl_nodes.to_crs(gu.projectedCrsNL)

In [ ]:
nl_node_kdtree, nl_node_coords, nl_node_ids = gu.buildRoadKDTree(nl_nodes_projected)

In [ ]:
nl_ferry_coordinates, nl_ferry_ids = gu.extractFerryCoordinatesAndIds(nl_ferries_projected)

In [ ]:
nl_connectedFerries = gu.connectFerryToRoad(
    ferryCoords=nl_ferry_coordinates,
    ferryIds=nl_ferry_ids,
    roadCoords=nl_node_coords,
    roadNodeIds=nl_node_ids,
    roadTree=nl_node_kdtree,
    maxDistance=5000.0,
    targetCRS=nl_ferries.crs,
    sourceCRS=nl_ferries_projected.crs
)

In [ ]:
nl_connectedFerries.ferryId.value_counts()

In [ ]:
nl_ferries_projected[
    nl_ferries_projected.id == nl_connectedFerries.ferryId.value_counts().idxmin()
].explore(
    tooltip=['id', 'name'],
    style_kwds={'color': 'red', 'weight': 8}
)

In [ ]:
m = nl_ferries.explore(
    color='blue',
    tooltip=['name', 'from', 'to'],
    style_kwds={'weight': 2}
)
nl_connectedFerries[nl_connectedFerries['distance'] <=0.5].explore( 
    m=m,
    color='red',
    tooltip=['ferryPoint', 'roadPoint', 'distance'],
    style_kwds={'weight': 8}
)
m

In [ ]:
coord_to_ferry_ids = gu.groupFerryIdsByCoordinate(nl_ferry_coordinates, nl_ferry_ids)

# Extract the lengths of lists
list_lengths = [len(ferry_ids) for ferry_ids in coord_to_ferry_ids.values()]

# Plotting the histogram
plt.hist(list_lengths, bins=range(1, max(list_lengths) + 2), edgecolor='black')
plt.xlabel('Number of Ferry IDs per Coordinate')
plt.ylabel('Frequency')
plt.title('Histogram of Ferry IDs List Lengths')
plt.show()

In [ ]:
# Identify the maximum length
max_length = max(list_lengths)

# Find the coordinate(s) with the longest list of ferry IDs
coords_with_max_length = [coord for coord, ids in coord_to_ferry_ids.items() if len(ids) == max_length]

# Extract and print the ferry IDs associated with the longest list
longest_ferry_id_lists = [coord_to_ferry_ids[coord] for coord in coords_with_max_length]

In [ ]:
max_length

In [ ]:
nl_ferries[nl_ferries.id.isin(longest_ferry_id_lists[0])]

In [ ]:
# Transformer from source CRS (e.g. EPSG:28992) to EPSG:4326
to_wgs84 = pyproj.Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True).transform

m = nl_ferries[nl_ferries.id.isin(longest_ferry_id_lists[0])].explore(
    tooltip=['id', 'name'],
    style_kwds={'weight': 2},
    color='blue',
    legend=False
)
for coord in coords_with_max_length:
    # Transform from (x, y) in meters to (lon, lat) in degrees
    lon, lat = to_wgs84(*coord)

    print(f"Coordinate: ({lat},{lon}), Ferry IDs: {coord_to_ferry_ids[coord]}")
    folium.Marker(
        location=(lat, lon),
        popup=f"Ferry IDs: {', '.join(map(str, coord_to_ferry_ids[coord]))}",
        icon=folium.Icon(color='red')
    ).add_to(m)
m

In [ ]:
ferry_edges = gu.generateZeroDistanceEdges(nl_connectedFerries)
ferry_edges['distance_m'] = ferry_edges['ferryId'].map( nl_ferries_projected.set_index('id').geometry.length.to_dict() )
nl_ferries[nl_ferries.id == ferry_edges.loc[ferry_edges.distance_m.idxmax()].ferryId].explore()

In [ ]:
nodes_df = nl_nodes[['id', 'geometry']].copy().rename(columns={'id': 'osm_id'})

# Create full list of node ids used
used_node_ids = pd.Index(pd.concat([nl_edges['u'], nl_edges['v'], ferry_edges['fromNodeId'], ferry_edges['toNodeId']]).unique())

assert set(nodes_df['osm_id']) == set(used_node_ids)

# Mapping
id_map = {nid: i for i, nid in enumerate(used_node_ids)}

nodes_df = nodes_df[nodes_df['osm_id'].isin(id_map)].copy()
nodes_df['id'] = nodes_df['osm_id'].map(id_map)

nodes_df = nodes_df.set_index('id',drop=False)

In [ ]:
nodes_df.shape, nl_nodes.shape

In [ ]:
land_edges = nl_edges[['u', 'v', 'length']].copy()
land_edges['from'] = land_edges['u'].map(id_map)
land_edges['to'] = land_edges['v'].map(id_map)
land_edges['weight'] = land_edges['length']

ferry_edges = ferry_edges[['fromNodeId', 'toNodeId', 'distance_m']].copy()
ferry_edges['from'] = ferry_edges['fromNodeId'].map(id_map)
ferry_edges['to'] = ferry_edges['toNodeId'].map(id_map)
ferry_edges['weight'] = ferry_edges['distance_m']

In [ ]:
edges_df = pd.concat(
    [   land_edges[['from', 'to', 'weight']], 
        ferry_edges[['from', 'to', 'weight']]
    ]
).rename(columns={'from': 'u', 'to': 'v', 'weight': 'length'})

In [ ]:
assert set(nodes_df.id) == ( set(edges_df.u) | set(edges_df.v) )

In [ ]:
nodes_df = gu.assignComponentsToNodes(nodes_df, edges_df)

In [ ]:
nodes_df.to_parquet(_data_path.joinpath('pandana_nodes.parquet'), index=False)
edges_df.to_parquet(_data_path.joinpath('pandana_edges.parquet'), index=False)

In [ ]:
gu.showNodesColoredPerComponentWithBasemap(nodes_df, width=800, height=600, file_name='after_connecting_components_with_ferries.png')